In [1]:
import hashlib
import json
import time
from typing import List, Dict, Any, Optional, Tuple
from ecdsa import SigningKey, VerifyingKey, SECP256k1, BadSignatureError
from collections import defaultdict
import copy


In [2]:
class CryptoUtils:
    
    @staticmethod
    def sha256(data: str) -> str:
        return hashlib.sha256(data.encode('utf-8')).hexdigest()
    
    @staticmethod
    def generate_keypair() -> Tuple[SigningKey, VerifyingKey]:
        private_key = SigningKey.generate(curve=SECP256k1)
        public_key = private_key.get_verifying_key()
        return private_key, public_key
    
    @staticmethod
    def sign_data(data: str, private_key: SigningKey) -> bytes:
        return private_key.sign(data.encode('utf-8'))
    
    @staticmethod
    def verify_signature(data: str, signature: bytes, 
                        public_key: VerifyingKey) -> bool:
        try:
            public_key.verify(signature, data.encode('utf-8'))
            return True
        except BadSignatureError:
            return False
    
    @staticmethod
    def public_key_to_address(public_key: VerifyingKey) -> str:
        pubkey_bytes = public_key.to_string()
        hash_result = hashlib.sha256(pubkey_bytes).hexdigest()
        return '0x' + hash_result[-40:]


In [ ]:
# class MerkleTree:
    
#     def __init__(self, transactions: List[Dict]):
#         self.transactions = transactions
#         self.leaves = [self._hash_transaction(tx) for tx in transactions]
#         self.root = self._build_tree(self.leaves) if self.leaves else None
    
#     def _hash_transaction(self, tx: Dict) -> str:
#         tx_string = json.dumps(tx, sort_keys=True)
#         return CryptoUtils.sha256(tx_string)
    
#     def _build_tree(self, nodes: List[str]) -> str:
#         if len(nodes) == 0:
#             return None
#         if len(nodes) == 1:
#             return nodes[0]
        
#         new_level = []
#         for i in range(0, len(nodes), 2):
#             left = nodes[i]
#             right = nodes[i + 1] if i + 1 < len(nodes) else nodes[i]
#             combined = left + right
#             new_level.append(CryptoUtils.sha256(combined))
        
#         return self._build_tree(new_level)
    
#     def get_root(self) -> Optional[str]:
#         return self.root
    
#     def get_proof(self, tx_index: int) -> List[Tuple[str, str]]:
#         if tx_index < 0 or tx_index >= len(self.leaves):
#             return []
        
#         proof = []
#         nodes = self.leaves.copy()
#         index = tx_index
        
#         while len(nodes) > 1:
#             if index % 2 == 0:
#                 sibling_index = index + 1 if index + 1 < len(nodes) else index
#                 sibling = nodes[sibling_index]
#                 position = 'right'
#             else:
#                 sibling_index = index - 1
#                 sibling = nodes[sibling_index]
#                 position = 'left'
            
#             proof.append((sibling, position))
            
#             new_nodes = []
#             for i in range(0, len(nodes), 2):
#                 left = nodes[i]
#                 right = nodes[i + 1] if i + 1 < len(nodes) else nodes[i]
#                 new_nodes.append(CryptoUtils.sha256(left + right))
            
#             nodes = new_nodes
#             index = index // 2
        
#         return proof
    
#     @staticmethod
#     def verify_proof(leaf_hash: str, proof: List[Tuple[str, str]], 
#                     root: str) -> bool:
#         current_hash = leaf_hash
        
#         for sibling_hash, position in proof:
#             if position == 'left':
#                 combined = sibling_hash + current_hash
#             else:
#                 combined = current_hash + sibling_hash
#             current_hash = CryptoUtils.sha256(combined)
        
#         return current_hash == root


In [4]:
class AlertTransaction:
    
    def __init__(self, node_id: str, alert_type: str,
                 detector_outputs: Dict[str, float],
                 features_summary: Dict[str, Any],
                 timestamp: float = None):
        self.tx_id = None
        self.node_id = node_id
        self.alert_type = alert_type
        self.timestamp = timestamp or time.time()
        self.detector_outputs = detector_outputs
        self.features_summary = features_summary
        self.signature = None
        self.signer_address = None
    
    def to_dict(self, include_signature: bool = True) -> Dict:
        data = {
            'tx_id': self.tx_id,
            'node_id': self.node_id,
            'alert_type': self.alert_type,
            'timestamp': self.timestamp,
            'detector_outputs': self.detector_outputs,
            'features_summary': self.features_summary,
            'signer_address': self.signer_address
        }
        
        if include_signature and self.signature:
            data['signature'] = self.signature.hex()
        
        return data
    
    def get_signing_data(self) -> str:
        data = self.to_dict(include_signature=False)
        del data['tx_id']
        return json.dumps(data, sort_keys=True)
    
    def sign(self, private_key: SigningKey) -> None:
        signing_data = self.get_signing_data()
        self.signature = CryptoUtils.sign_data(signing_data, private_key)
        
        self.tx_id = CryptoUtils.sha256(signing_data + self.signature.hex())
        
        public_key = private_key.get_verifying_key()
        self.signer_address = CryptoUtils.public_key_to_address(public_key)
    
    def verify(self, public_key: VerifyingKey) -> bool:
        if not self.signature:
            return False
        
        signing_data = self.get_signing_data()
        return CryptoUtils.verify_signature(signing_data, self.signature, public_key)


In [ ]:
class Hashed_Block:
    
    def __init__(self, block ):
        self.block_number = block.block_number
        self.timestamp = block.time_stamp
        self.previous_hash = block.previous_hash
        self.proposer_id = block.proposer_id
        self.block_hash = block.block_hash
        

In [ ]:
class Block:
    
    def __init__(self, block_number: int,
                 previous_hash: str, 
                #  view_number: int = 0,
                #  sequence_number: int = 0, 
                 proposer_id: str = None,
                 timestamp: float = None,

                 metadata =[]
                 ):
        self.block_number = block_number
        self.timestamp = time.time()
        self.previous_hash = previous_hash
        self.metadata = metadata 
            #   Merkle 
        # self.transactions = transactions
        
        # self.view_number = view_number
        # self.sequence_number = sequence_number
        self.proposer_id = proposer_id
        
        self.prepare_signatures = []
        self.commit_signatures = []
        
        # tx_dicts = [tx.to_dict() for tx in transactions]
        # self.merkle_tree = MerkleTree(tx_dicts)
        # self.merkle_root = self.merkle_tree.get_root() or "0" * 64
        self.block_hash = self.calculate_hash()

    
    def calculate_hash(self) -> str:
        header_data = {
            'block_number': self.block_number,
            'timestamp': self.timestamp,
            'previous_hash': self.previous_hash,
            # 'merkle_root': self.merkle_root,
            # 'view_number': self.view_number,
            # 'sequence_number': self.sequence_number,
            'meta_data' : self.metadata,
            'proposer_id': self.proposer_id,

        }
        
        header_string = json.dumps(header_data, sort_keys=True)
        return CryptoUtils.sha256(header_string)
    
    def add_prepare_signature(self, validator_id: str, signature: bytes):
        self.prepare_signatures.append({
            'validator_id': validator_id,
            'signature': signature.hex()
        })
    
    def add_commit_signature(self, validator_id: str, signature: bytes):
        self.commit_signatures.append({
            'validator_id': validator_id,
            'signature': signature.hex()
        })
    
    # def to_dict(self) -> Dict:
    #     return {
    #         'block_number': self.block_number,
    #         'timestamp': self.timestamp,
    #         'previous_hash': self.previous_hash,
    #         'merkle_root': self.merkle_root,
    #         'view_number': self.view_number,
    #         'sequence_number': self.sequence_number,
    #         'proposer_id': self.proposer_id,
    #         'transactions': [tx.to_dict() for tx in self.transactions],
    #         'prepare_signatures': self.prepare_signatures,
    #         'commit_signatures': self.commit_signatures,
    #         'alert_count': self.alert_count,
    #         'severity_summary': self.severity_summary,
    #         'block_hash': self.block_hash
    #     }
    
    # def verify_transaction_inclusion(self, tx_index: int) -> bool:
    #     if tx_index < 0 or tx_index >= len(self.transactions):
    #         return False
        
    #     tx = self.transactions[tx_index]
    #     leaf_hash = self.merkle_tree._hash_transaction(tx.to_dict())
    #     proof = self.merkle_tree.get_proof(tx_index)
        
    #     return MerkleTree.verify_proof(leaf_hash, proof, self.merkle_root)


In [ ]:
class Blockchain:
    
    def __init__(self):
        self.chain: List[Block] = []
        self.pending_transactions: List[AlertTransaction] = []
        # self.validators: Dict[str, VerifyingKey] = {}
        # self.current_view = 0
        # self.current_sequence = 0
        
        self._create_genesis_block()
    
    def _create_genesis_block(self) -> None:
        genesis = Block(
            block_number=0,
            # transactions=[],
            previous_hash="0" * 64,
            # view_number=0,
            # sequence_number=0,
            proposer_id="genesis",
            metadata = []
        )
        self.chain.append(genesis)
    
    # def register_validator(self, validator_id: str, public_key: VerifyingKey):
    #     self.validators[validator_id] = public_key
    
    def add_transaction(self, transaction: AlertTransaction) -> bool:
        self.pending_transactions.append(transaction)
        return True
    
    def create_block(self, proposer_id: str, max_transactions: int = 100) -> Block:
        if not self.pending_transactions:
            return None
        
        # transactions = self.pending_transactions[:max_transactions]
        
        previous_block = self.chain[-1]
        new_block = Block(
            block_number=len(self.chain),
            transactions=transactions,
            previous_hash=previous_block.block_hash,
            view_number=self.current_view,
            sequence_number=self.current_sequence,
            proposer_id=proposer_id
        )
        
        self.current_sequence += 1
        return new_block
    
    def add_block(self, block: Block) -> bool:

        if not self.validate_chain(): return "Chain tampered"

        if block.block_number != len(self.chain):
            print(f"Invalid block number: {block.block_number}, expected {len(self.chain)}")
            return False
        
        expected_hash = block.calculate_hash()
        if block.block_hash != expected_hash:
            print("Invalid block hash")
            return False
        
        self.chain.append(block)
        
        # tx_ids = {tx.tx_id for tx in block.transactions}
        # self.pending_transactions = [
        #     tx for tx in self.pending_transactions 
        #     if tx.tx_id not in tx_ids
        # ]
        
        return True
    
    def validate_chain(self) -> bool:
        if len(self.chain) == 0:
            return False
        
        genesis = self.chain[0]
        if genesis.block_number != 0 or genesis.previous_hash != "0" * 64:
            return False
        
        for i in range(1, len(self.chain)):
            current_block = self.chain[i]
            previous_block = self.chain[i - 1]
            
            if current_block.block_number != i:
                return False
            
            if current_block.previous_hash != previous_block.block_hash:
                return False
            
            if current_block.block_hash != current_block.calculate_hash():
                return False
        
        return True
    
    def get_block(self, block_number: int) -> Optional[Block]:
        if 0 <= block_number < len(self.chain):
            return self.chain[block_number]
        return None
    
    def get_latest_block(self) -> Block:
        return self.chain[-1]
  

In [ ]:
     
    # def get_alerts_by_type(self, alert_type: str) -> List[AlertTransaction]:
    #     alerts = []
    #     for block in self.chain[1:]:
    #         # metadata = Dict 
    #         if block.metadata.alert_type  == alert_type
    #         # if tx.alert_type == alert_type:
    #             alerts.append(tx)
    #     return alerts
    
    # def get_alerts_by_node(self, node_id: str) -> List[AlertTransaction]:
    #     alerts = []
    #     for block in self.chain[1:]:
    #         for tx in block.transactions:
    #             if tx.node_id == node_id:
    #                 alerts.append(tx)
    #     return alerts
    
    # def get_statistics(self) -> Dict[str, Any]:
    #     total_alerts = sum(block.alert_count for block in self.chain[1:])
        
    #     severity_totals = {'low': 0, 'medium': 0, 'high': 0, 'critical': 0}
    #     for block in self.chain[1:]:
    #         for severity in severity_totals:
    #             severity_totals[severity] += block.severity_summary[severity]
        
    #     return {
    #         'total_blocks': len(self.chain),
    #         'total_alerts': total_alerts,
    #         'pending_transactions': len(self.pending_transactions),
    #         'severity_summary': severity_totals,
    #         'chain_valid': self.validate_chain()
    #     }


In [ ]:
def demo_blockchain():
    print("\n" + "="*70)
    print("IDS BLOCKCHAIN ")
    print("="*70 + "\n")
    
    blockchain = IDSBlockchain()
    print("Blockchain initialized with genesis block")
    
    node1_private, node1_public = CryptoUtils.generate_keypair()
    node1_address = CryptoUtils.public_key_to_address(node1_public)
    
    node2_private, node2_public = CryptoUtils.generate_keypair()
    node2_address = CryptoUtils.public_key_to_address(node2_public)
    
    print(f"Generated keys for Node 1: {node1_address[:20]}...")
    print(f"Generated keys for Node 2: {node2_address[:20]}...")
    
    alert1 = AlertTransaction(
        node_id="node_1",
        alert_type="DDoS",
        detector_outputs={"snort": 1.0, "rf": 0.89, "if": 0.42},
        features_summary={"src_ip_hash": "0xabc123", "dst_port": 80, "packet_count": 10000}
    )
    alert1.sign(node1_private)
    print(f"\nCreated Alert 1: {alert1.alert_type} (tx_id: {alert1.tx_id[:16]}...)")
    
    alert2 = AlertTransaction(
        node_id="node_2",
        alert_type="port_scan",
        detector_outputs={"snort": 1.0, "rf": 0.65},
        features_summary={"src_ip_hash": "0xdef456", "dst_ports": [22, 80, 443, 8080]}
    )
    alert2.sign(node2_private)
    print(f"Created Alert 2: {alert2.alert_type} (tx_id: {alert2.tx_id[:16]}...)")
    
    alert3 = AlertTransaction(
        node_id="node_1",
        alert_type="data_exfiltration",
        detector_outputs={"dlp": 0.95, "ml": 0.88},
        features_summary={"dst_ip_hash": "0x789abc", "data_volume_mb": 500}
    )
    alert3.sign(node1_private)
    print(f"Created Alert 3: {alert3.alert_type} (tx_id: {alert3.tx_id[:16]}...)")
    
    blockchain.add_transaction(alert1)
    blockchain.add_transaction(alert2)
    blockchain.add_transaction(alert3)
    print(f"\nAdded {len(blockchain.pending_transactions)} transactions to pending pool")
    
    new_block = blockchain.create_block(proposer_id=node1_address, max_transactions=10)
    print(f"\nCreated Block #{new_block.block_number}")
    print(f"  - Timestamp: {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(new_block.timestamp))}")
    print(f"  - Transactions: {new_block.alert_count}")
    print(f"  - Merkle Root: {new_block.merkle_root[:32]}...")
    print(f"  - Block Hash: {new_block.block_hash[:32]}...")
    print(f"  - Severity: {new_block.severity_summary}")
    
    blockchain.add_block(new_block)
    print(f"\nBlock added to chain")
    
    is_valid = blockchain.validate_chain()
    print(f"\nChain validation: {'VALID' if is_valid else 'INVALID'}")
    
    stats = blockchain.get_statistics()
    print(f"\nBlockchain Statistics:")
    print(f"  - Total Blocks: {stats['total_blocks']}")
    print(f"  - Total Alerts: {stats['total_alerts']}")
    print(f"  - Pending Transactions: {stats['pending_transactions']}")
    print(f"  - Severity Summary: {stats['severity_summary']}")
    print("\n" + "="*70 + "\n")


In [ ]:
from collections import defaultdict

class PBFT_Node:
    """
    This class represents A Node participating in the PBFT consensus mechanism.

    Parameters:
    Node_Id     -> int (Represents the Node's id)
    Total_Nodes -> int (Represents the total number of Nodes participating in the consensus)
    F           -> int (Represents the maximum faulty nodes)
    Is_Primary  -> bool (True if the Node is a Primary Node)
    
    """

    def __init__(self, Node_Id, Total_Nodes, F, Is_Primary=False):
        self.Node_Id = Node_Id
        self.Total_Nodes = Total_Nodes
        self.F = F
        self.Is_Primary = Is_Primary

        self.View = 0
        self.Seq = 0

        self.Pre_Prepare = defaultdict(lambda: defaultdict(int))
        self.Prepare = defaultdict(lambda: defaultdict(set))
        self.Commit = defaultdict(lambda: defaultdict(set))
        self.Committed = {}
    
    def Is_Valid_Block(self,Block_Instance):
        # Passes the Block_Instance to the IDS and returns True is the block_signature is identified as malicious
        return True
    
    def Propose(self, meta_data):
        Block_instance = Block(meta_data)
       

        return {
            "Type": "PRE-PREPARE",
            "View": self.View,
            "Seq": self.Seq,
            "Block_Instance": Block_Instance,
            "Block_Hash": Block_Hash,
            "Sender": self.Node_Id
        }

    def Receive(self, Msg):
        if Msg["View"] != self.View:
            return None

        if Msg["Type"] == "PRE-PREPARE":
            return self._On_Pre_Prepare(Msg)

        if Msg["Type"] == "PREPARE":
            return self._On_Prepare(Msg)

        if Msg["Type"] == "COMMIT":
            return self._On_Commit(Msg)

        return None

    def _On_Pre_Prepare(self, Msg):
        Seq = Msg["Seq"]
        Block_Instance = Msg["Block_Instance"]
        Block_Hash = Msg["Block_Hash"]

        if Seq in self.Pre_Prepare:
            return None

        if Block_Instance.calculate_hash() != Block_Hash:
            return None
        
        # This is the only phase in the consensus mechanism where the IDS part comes into play. 
        if not self.Is_Valid_Block(Block_Instance):
            return None
        
        self.Pre_Prepare[Seq]["Block_Hash"] = Block_Hash

        return {
            "Type": "PREPARE",
            "View": Msg["View"],
            "Seq": Seq,
            "Block_Hash": Block_Hash,
            "Sender": self.Node_Id
        }

    def _On_Prepare(self, Msg):
        Seq = Msg["Seq"]
        Block_Hash = Msg["Block_Hash"]
        Sender = Msg["Sender"]

        if Seq not in self.Pre_Prepare:
            return None

        if self.Pre_Prepare[Seq]["Block_Hash"] != Block_Hash:
            return None

        self.Prepare[Seq][Block_Hash].add(Sender)

        if len(self.Prepare[Seq][Block_Hash]) >= 2 * self.F + 1:
            return {
                "Type": "COMMIT",
                "View": Msg["View"],
                "Seq": Seq,
                "Block_Hash": Block_Hash,
                "Sender": self.Node_Id
            }

        return None

    def _On_Commit(self, Msg):
        Seq = Msg["Seq"]
        Block_Hash = Msg["Block_Hash"]
        Sender = Msg["Sender"]

        self.Commit[Seq][Block_Hash].add(Sender)

        if len(self.Commit[Seq][Block_Hash]) >= 2 * self.F + 1:
            if Seq not in self.Committed:
                self.Committed[Seq] = Block_Hash
                return {
                    "Type": "DECIDED",
                    "Seq": Seq,
                    "Block_Hash": Block_Hash
                }

        return None

    # -------------------------------------------------
    # Networking Hook Methods (EMPTY BY DESIGN)
    # -------------------------------------------------

    def Send_To_All_Nodes(self, Msg):
        """
        Broadcast a PBFT message to all other nodes.
        Networking layer will implement this.
        """
        pass

    # def Send_To_Primary(self, Msg):
    #     """
    #     Send a message directly to the current primary.
    #     Useful for view-change or client replies.
    #     """
    #     pass

    def On_Message_Received_From_Network(self, Msg):
        """
        Called by the networking layer when a message arrives.
        Should call Receive() and forward any returned message.
        """
        pass

    def On_Block_Committed(self, Seq, Block_Hash):
        """
        Callback invoked when a block reaches DECIDED state.
        Used to update blockchain state or notify IDS.
        """
        pass

In [ ]:
from collections import defaultdict

class PBFT_Node:
    """
    PBFT Node with:
    - Deterministic leader selection (view % N)
    - IDS-based verification in PRE-PREPARE
    - Network-safe message format
    """

    def __init__(self, Node_Id, Total_Nodes, F, ids_model=None):
        """
        Node_Id      : int (0 ... N-1)
        Total_Nodes  : int
        F            : max faulty nodes
        ids_model    : callable(alert_metadata) -> bool
        """
        self.Node_Id = Node_Id
        self.Total_Nodes = Total_Nodes
        self.F = F

        # PBFT state
        self.View = 0
        self.Seq = 0

        self.Pre_Prepare = {}                          # Seq -> {Block_Hash, Alert_Metadata}
        self.Prepare = defaultdict(lambda: defaultdict(set))  # Seq -> hash -> voters
        self.Commit = defaultdict(lambda: defaultdict(set))   # Seq -> hash -> voters
        self.Committed = {}                            # Seq -> Block_Hash

        # Local block storage (never sent over network)
        self.Block_Pool = {}                           # Block_Hash -> Block

        # IDS model
        self.ids_model = ids_model


    def Is_Primary_Node(self) -> bool:
        """
        Deterministic PBFT leader selection
        """
        return self.Node_Id == (self.View % self.Total_Nodes)

    def Get_Current_Leader(self) -> int:
        return self.View % self.Total_Nodes


    def Is_Valid_Alert(self, alert_metadata: dict) -> bool:
        """
        Run alert metadata on local IDS.
        Returns True if malicious.
        """
        if self.ids_model is None:
            return True 
        return self.ids_model(alert_metadata)


    def Propose(self, block):
        if not self.Is_Primary_Node():
            raise Exception("Only primary can propose")

        self.Seq += 1
        block_hash = block.block_hash

        self.Block_Pool[block_hash] = block

        alert_metadata = [tx.to_dict() for tx in block.transactions]

        # Record pre-prepare locally
        self.Pre_Prepare[self.Seq] = {
            "Block_Hash": block_hash,
            "Alert_Metadata": alert_metadata
        }

        return {
            "Type": "PRE-PREPARE",
            "View": self.View,
            "Seq": self.Seq,
            "Block_Hash": block_hash,
            "Alert_Metadata": alert_metadata,
            "Sender": self.Node_Id
        }


    def On_Message_Received_From_Network(self, msg):
        response = self.Receive(msg)

        if response:
            if response["Type"] == "DECIDED":
                self.On_Block_Committed(
                    response["Seq"],
                    response["Block_Hash"]
                )
            else:
                self.Send_To_All_Nodes(response)


    def Receive(self, Msg):
        if Msg["View"] != self.View:
            return None

        if Msg["Type"] == "PRE-PREPARE":
            return self._On_Pre_Prepare(Msg)

        if Msg["Type"] == "PREPARE":
            return self._On_Prepare(Msg)

        if Msg["Type"] == "COMMIT":
            return self._On_Commit(Msg)

        return None

    def _On_Pre_Prepare(self, Msg):
        Seq = Msg["Seq"]
        block_hash = Msg["Block_Hash"]
        alert_metadata = Msg["Alert_Metadata"]

        if Seq in self.Pre_Prepare:
            return None

        if not self.Is_Valid_Alert(alert_metadata):
            return None

        self.Pre_Prepare[Seq] = {
            "Block_Hash": block_hash,
            "Alert_Metadata": alert_metadata
        }

        return {
            "Type": "PREPARE",
            "View": Msg["View"],
            "Seq": Seq,
            "Block_Hash": block_hash,
            "Sender": self.Node_Id
        }

    def _On_Prepare(self, Msg):
        Seq = Msg["Seq"]
        block_hash = Msg["Block_Hash"]
        sender = Msg["Sender"]

        if Seq not in self.Pre_Prepare:
            return None

        self.Prepare[Seq][block_hash].add(sender)

        if len(self.Prepare[Seq][block_hash]) >= 2 * self.F + 1:
            return {
                "Type": "COMMIT",
                "View": Msg["View"],
                "Seq": Seq,
                "Block_Hash": block_hash,
                "Sender": self.Node_Id
            }

        return None

    def _On_Commit(self, Msg):
        Seq = Msg["Seq"]
        block_hash = Msg["Block_Hash"]
        sender = Msg["Sender"]

        self.Commit[Seq][block_hash].add(sender)

        if len(self.Commit[Seq][block_hash]) >= 2 * self.F + 1:
            if Seq not in self.Committed:
                self.Committed[Seq] = block_hash
                return {
                    "Type": "DECIDED",
                    "Seq": Seq,
                    "Block_Hash": block_hash
                }

        return None

    def Send_To_All_Nodes(self, Msg):
        """
        Implemented by networking layer
        """
        pass

    def On_Block_Committed(self, Seq, Block_Hash):
        """
        Implemented by node controller
        """
        pass


In [ ]:

from flask import Flask, request, jsonify

class NodeAPI:
    def __init__(self, pbft_node):
        self.pbft = pbft_node
        self.app = Flask(__name__)
        self._register_routes()


    def _register_routes(self):

        @self.app.route("/health", methods=["GET"])
        def health():
            return jsonify({"status": "ok"}), 200

        @self.app.route("/pbft", methods=["POST"])
        def receive_pbft():
            msg = request.json

            if not msg or "Type" not in msg:
                return jsonify({"error": "Invalid PBFT message"}), 400

            self.pbft.On_Message_Received_From_Network(msg)
            return jsonify({"status": "received"}), 200

    def start(self, port):
        print(f"[NodeAPI] Listening on port {port}")
        self.app.run(host="0.0.0.0", port=port, threaded=True)


In [ ]:

import requests

class PeerClient:
    def __init__(self, peers):
        """
        peers: dict {node_id: base_url}
        """
        self.peers = peers

    def send(self, node_id, msg):
        try:
            requests.post(
                f"{self.peers[node_id]}/pbft",
                json=msg,
                timeout=2
            )
        except requests.RequestException:
            pass

    def broadcast(self, msg):
        for node_id in self.peers:
            self.send(node_id, msg)


In [ ]:

class Node:
    def __init__(self, node_id, port, peers,
                 blockchain, pbft_node):

        """
        node_id    : int (0 .. N-1)
        peers      : dict {node_id: url}
        """
        self.node_id = node_id
        self.blockchain = blockchain
        self.pbft = pbft_node
        self.peer_client = PeerClient(peers)
        self.port = port

        self.pbft.Send_To_All_Nodes = self._send_to_all
        self.pbft.On_Message_Received_From_Network = self._on_message
        self.pbft.On_Block_Committed = self._on_block_committed

        self.api = NodeAPI(self.pbft)

    def _send_to_all(self, msg):
        print(f"[Node {self.node_id}] Broadcasting {msg['Type']}")
        self.peer_client.broadcast(msg)

    def _on_message(self, msg):
        """
        Called by NodeAPI when a message arrives.
        """
        response = self.pbft.Receive(msg)

        if response:
            if response["Type"] == "DECIDED":
                self.pbft.On_Block_Committed(
                    response["Seq"],
                    response["Block_Hash"]
                )
            else:
                self.pbft.Send_To_All_Nodes(response)

    def _on_block_committed(self, seq, block_hash):
        """
        Final PBFT decision → blockchain commit
        """
        block = self.pbft.Block_Pool.get(block_hash)

        if block is None:
            print(f"[Node {self.node_id}] Block missing for commit")
            return

        success = self.blockchain.add_block(block)

        if success:
            print(f"[Node {self.node_id}] Block committed (seq={seq})")
        else:
            print(f"[Node {self.node_id}] Block rejected by blockchain")


    def start(self):
        print(
            f"[Node {self.node_id}] Starting | "
            f"Leader = {self.pbft.Get_Current_Leader()} | "
            f"Port = {self.port}"
        )
        self.api.start(self.port)


In [ ]:
# module3/config.py
def get_config(node_id):
    configs = {
        0: {
            "port": 5000,
            "peers": {
                1: "http://localhost:5001",
                2: "http://localhost:5002"
            }
        },
        1: {
            "port": 5001,
            "peers": {
                0: "http://localhost:5000",
                2: "http://localhost:5002"
            }
        },
        2: {
            "port": 5002,
            "peers": {
                0: "http://localhost:5000",
                1: "http://localhost:5001"
            }
        }
    }
    return configs[node_id]


In [ ]:
# validator = node id


# local copy of all the alerts
# data storage
# buffers
# block with the alert features